# Stress test: can QPOP's design find the best drug pair?

**Real:** the experimental design. 155 runs, 12 drugs, 3 levels (0 = none, 1 = low, 2 = high), copied from Goh et al. 2022, Table S8 (`data/goh2022_lymphoma_tableS8_oacd_design.csv`). The same 155-run, 12-drug, 3-level scheme is used in the AML 2024 and lymphoma 2025 papers.
**Simulated:** the patient's true response. Each drug kills along its own dose curve, drugs combine independently (Bliss), and 3 random pairs get extra synergy. Measurement noise is added to each run.

QPOP fits a full quadratic (1 + 12 + 12 + 66 = 91 terms) to the 155 runs and ranks all doublets. The clinical papers offer the top three doublets to the doctor (Goh 2022: "ranked among the top three doublet combinations").
Comparison: a simpler model with no drug-drug terms (25 terms), and a random pick.
QPOP itself uses stepwise term removal (MATLAB `LinearModel.stepwise`, Rashid 2018 data file S1); this notebook fits all terms by least squares, which is a simplification.

In [1]:
import csv, itertools
import numpy as np
import plotly.graph_objects as go

DESIGN = np.array([[int(x) for x in r[1:]] for r in list(csv.reader(open("../data/goh2022_lymphoma_tableS8_oacd_design.csv")))[1:]], float)
K = DESIGN.shape[1]
PAIRS = list(itertools.combinations(range(K), 2))
print("runs:", DESIGN.shape[0], "drugs:", K, "| runs with each drug at the low level:", (DESIGN == 1).sum(0))

def feats(Z, inter=True):
    cols = [np.ones(len(Z))] + [Z[:, j] for j in range(K)] + [Z[:, j] ** 2 for j in range(K)]
    if inter:
        cols += [Z[:, i] * Z[:, j] for i, j in PAIRS]
    return np.stack(cols, 1)

DOUBLETS = []
for i, j in PAIRS:
    for a in (1, 2):
        for b in (1, 2):
            z = np.zeros(K); z[i] = a; z[j] = b
            DOUBLETS.append(z)
DOUBLETS = np.array(DOUBLETS)
PAIR_OF = np.repeat(np.arange(len(PAIRS)), 4)
P_Q, P_M = np.linalg.pinv(feats(DESIGN)), np.linalg.pinv(feats(DESIGN, False))
F_Q, F_M = feats(DOUBLETS), feats(DOUBLETS, False)
print("doublet options ranked:", len(DOUBLETS), "| quadratic terms:", F_Q.shape[1], "| no-interaction terms:", F_M.shape[1])

runs: 155 drugs: 12 | runs with each drug at the low level: [9 9 9 9 9 9 9 9 9 9 9 9]
doublet options ranked: 264 | quadratic terms: 91 | no-interaction terms: 25


In [2]:
rng = np.random.default_rng(0)

def truth(Z, E, H, S):
    """True viability. E: max kill per drug, H: low-dose kill as share of max, S: {pair: synergy}."""
    kill = np.where(Z == 2, E, np.where(Z == 1, E * H, 0.0))
    v = np.prod(1 - kill, axis=-1)
    a = Z / 2
    for (i, j), s in S.items():
        v = v * np.exp(-s * a[..., i] * a[..., j])
    return v

def run(sd, syn, n=400):
    out = np.zeros((3, 4))          # rows: quadratic, no-interaction, random; cols: exact, top3, right pair, regret
    for _ in range(n):
        E, H = rng.uniform(0, 0.6, K), rng.uniform(0.1, 0.6, K)
        S = {PAIRS[k]: syn for k in rng.choice(len(PAIRS), 3, replace=False)}
        y = truth(DESIGN, E, H, S) + rng.normal(0, sd, len(DESIGN))
        tv = truth(DOUBLETS, E, H, S); tb = tv.argmin()
        for r, pred in enumerate((F_Q @ (P_Q @ y), F_M @ (P_M @ y), rng.random(len(DOUBLETS)))):
            c = pred.argmin()
            out[r] += [c == tb, tb in np.argsort(pred)[:3], PAIR_OF[c] == PAIR_OF[tb], tv[c] - tv[tb]]
    return out / n

NOISE = np.array([0, 0.025, 0.05, 0.075, 0.1, 0.15])
SYN = np.array([0, 0.25, 0.5, 0.75, 1.0])
RES = np.array([[run(sd, s) for sd in NOISE] for s in SYN])   # (syn, noise, model, metric)
print("random pick, top-3 hit rate:", round(RES[..., 2, 1].mean(), 3), "(expected 3/264 =", round(3 / 264, 3), ")")

random pick, top-3 hit rate: 0.01 (expected 3/264 = 0.011 )


## 1. Top-3 hit rate: is the true best doublet among the three QPOP would report? (rotate)

In [3]:
fig = go.Figure()
fig.add_trace(go.Surface(x=NOISE, y=SYN, z=RES[..., 0, 1], colorscale="Viridis", opacity=0.9, name="QPOP quadratic",
                         colorbar=dict(title="hit rate")))
fig.add_trace(go.Surface(x=NOISE, y=SYN, z=RES[..., 1, 1], colorscale="Greys", opacity=0.6, showscale=False,
                         name="no interactions"))
fig.update_layout(scene=dict(xaxis_title="noise SD (viability)", yaxis_title="synergy strength",
                             zaxis_title="top-3 hit rate", zaxis=dict(range=[0, 1])),
                  title="Colour: QPOP quadratic. Grey: model without drug-drug terms", height=620)
fig

## 2. At moderate synergy (0.5), by noise

The shaded band marks noise SD 0.05-0.10. Lee 2025 accepted assay coefficients of variation up to 20%; at 50% viability that is an SD of about 0.1. The noise per run in the real assay is not reported, so this band is an assumption.

In [4]:
j = list(SYN).index(0.5)
fig = go.Figure()
for r, name in enumerate(["QPOP quadratic", "no interactions", "random"]):
    fig.add_trace(go.Scatter(x=NOISE, y=RES[j, :, r, 1], mode="lines+markers", name=f"{name}: top-3"))
fig.add_trace(go.Scatter(x=NOISE, y=RES[j, :, 0, 2], mode="lines+markers", line=dict(dash="dash"),
                         name="QPOP: right pair (any dose)"))
fig.add_vrect(x0=0.05, x1=0.10, opacity=0.12, line_width=0)
fig.update_layout(xaxis_title="noise SD", yaxis=dict(title="rate", range=[0, 1]), height=430,
                  title="Synergy 0.5: hit rates by noise")
fig

In [5]:
print("synergy  noise   QPOP exact  QPOP top3  QPOP pair  no-inter top3  QPOP regret")
for a, s in enumerate(SYN):
    for b, sd in enumerate(NOISE):
        q, m = RES[a, b, 0], RES[a, b, 1]
        print(f"{s:6.2f} {sd:7.3f} {q[0]:10.2f} {q[1]:10.2f} {q[2]:10.2f} {m[1]:13.2f} {q[3]:12.3f}")

synergy  noise   QPOP exact  QPOP top3  QPOP pair  no-inter top3  QPOP regret
  0.00   0.000       0.90       0.98       0.90          0.98        0.000
  0.00   0.025       0.60       0.92       0.60          0.93        0.007
  0.00   0.050       0.46       0.76       0.46          0.79        0.018
  0.00   0.075       0.33       0.63       0.33          0.64        0.033
  0.00   0.100       0.21       0.47       0.21          0.49        0.055
  0.00   0.150       0.16       0.32       0.17          0.29        0.085
  0.25   0.000       0.74       0.96       0.74          0.87        0.003
  0.25   0.025       0.56       0.89       0.56          0.78        0.010
  0.25   0.050       0.44       0.74       0.44          0.65        0.022
  0.25   0.075       0.29       0.59       0.29          0.50        0.039
  0.25   0.100       0.20       0.45       0.21          0.34        0.056
  0.25   0.150       0.14       0.32       0.15          0.16        0.096
  0.50   0.000       0

## What this says

1. **The design and model are reasonable when noise is low.** With no noise, the true best doublet is in QPOP's top three in most simulated patients, and far above a random pick (about 1%).
2. **Drug-drug terms matter.** When real synergy exists, the full quadratic clearly beats a model without interaction terms. This is QPOP's genuine advantage.
3. **Noise is the weak point.** In the plausible noise band (SD 0.05-0.10) the top-3 hit rate falls to about 40-75%, and the exact best doublet is found only 20-45% of the time. The misses are wrong *pairs*: "right pair" and "exact" are almost equal.
4. **Dose is not really tested here.** In this truth model more drug always kills more and there is no toxicity term, so the high dose always wins. Each drug is tested at its low dose in only 9 of 155 runs, so a truth with a real dose optimum would be harder.
5. **What this does not test.** Higher-order (3-drug) interactions, the stepwise term removal QPOP actually uses, cells that differ from the patient's tumour in the body, and whether lab viability predicts clinical response. The clinical papers are the test of that last point, and their evidence is weak (`qpop_evidence.ipynb`).

All of this rests on a made-up truth model. It shows what the design *can* do, not what it does in patients.